# Step 02: Detect and Audit Dataset Anomalies
### Automated 8-Rule Quality Inspector, Multi-Panel Diagnostic Graphs & Visual Sample Audits
Directly integrated with `multi_class_train_rfdetr.ipynb`'s Step B verification (Cell 12).

### 8 Anomaly Detection Rules:
1. **Degenerate Boxes**: Width or height <= 0.
2. **Micro-Clicks**: Accidental clicks with width/height < 4px or area < 16px.
3. **Giant Boxes**: Oversized bounding boxes covering > 95% of image area.
4. **Extreme Aspect Ratios**: Ultra-thin slivers ($w/h < 0.1$ or $w/h > 10$).
5. **Out-of-Bounds Boxes**: Coordinates exceeding image width/height boundaries.
6. **Exact Duplicates**: Overlapping boxes of the same class with IoU > 0.95.
7. **Loose Margin Padding**: Excessive background padding around tags.
8. **Dense Query Overlaps**: Overly clustered overlapping tags.

### Key Outputs:
- Multi-Panel Anomaly Diagnostic Graphs (Distribution, Aspect Ratios, Areas, Density).
- Visual Sample Inspector displaying flagged anomalies (red boxes & yellow reason tags) vs clean boxes (green).
- `anomalies_manifest.json`: Complete record of all flagged anomaly boxes.
- `cleaned_dataset/`: Sanitized COCO annotations strictly excluding all anomalies.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)


import io
def show_plt_figure(fig=None):
    """Guarantees inline plot display in Jupyter under headless and Agg backends."""
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    display(pil_img)

In [ ]:
# CELL 2 - Paths, Output Setup & Field Configuration
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)

# COCO annotation field mappings
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

OUTPUT_ANOMALIES_DIR = Path("anomalies_audit")
OUTPUT_ANOMALIES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATASET_DIR = Path("cleaned_dataset")
CLEANED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Input JSON Dir:        {INPUT_JSON_DIR.resolve()}")
logger.info(f"Active Images Pool:    {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")
logger.info(f"Anomalies Output Dir:  {OUTPUT_ANOMALIES_DIR.resolve()}")
logger.info(f"Cleaned Dataset Dir:   {CLEANED_DATASET_DIR.resolve()}")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in json_files:
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in raw_records:
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")
# Construct definitions for downstream sanitized dataset exports
categories_list = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]
annotations_by_image = image_annotations
annotations_by_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
for anns in image_annotations.values():
    for a in anns:
        annotations_by_cat[a["category_name"]].append(a)

In [ ]:
# CELL 5 - 8-Rule Automated Anomaly Detector Engine
def compute_box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    denom = areaA + areaB - interArea
    return interArea / denom if denom > 0 else 0.0

flagged_anomalies = []
clean_annotations = []

# Use annotations_by_image or fallback to image_annotations
image_source = annotations_by_image if 'annotations_by_image' in globals() else image_annotations
for img_url, anns in image_source.items():
    img_record = next((r for r in raw_records if (r.get("coco_url") == img_url or r.get("url") == img_url)), {})
    img_w = float(img_record.get("width", 1920) or 1920)
    img_h = float(img_record.get("height", 1080) or 1080)
    
    for i, a in enumerate(anns):
        bbox = a.get("bbox", [])
        if len(bbox) != 4:
            flagged_anomalies.append({"annotation": a, "image_url": img_url, "reasons": ["Malformed bbox dimensions"], "bbox": [0,0,0,0]})
            continue
            
        x, y, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
        area = w * h
        aspect = w / max(1.0, h)
        is_anomaly = False
        reasons = []
        
        # Rule 1: Degenerate boxes
        if w <= 0 or h <= 0:
            is_anomaly = True
            reasons.append(f"Degenerate box (w={w}, h={h})")
            
        # Rule 2: Micro-clicks
        if w < 4 or h < 4 or area < 16:
            is_anomaly = True
            reasons.append(f"Micro-click (w={w:.1f}, h={h:.1f}, area={area:.1f})")
            
        # Rule 3: Giant boxes
        if w > 0.95 * img_w and h > 0.95 * img_h:
            is_anomaly = True
            reasons.append(f"Giant box covering >95% image (w={w:.1f}, h={h:.1f})")
            
        # Rule 4: Extreme aspect ratio slivers
        if aspect < 0.08 or aspect > 12.0:
            is_anomaly = True
            reasons.append(f"Extreme aspect ratio sliver ({aspect:.2f})")
            
        # Rule 5: Out of bounds
        if x < -5 or y < -5 or (x + w) > (img_w + 10) or (y + h) > (img_h + 10):
            is_anomaly = True
            reasons.append("Out of bounds coordinate")
            
        # Rule 6: Duplicate overlap
        for j, b in enumerate(anns):
            if i != j and a.get("category_name") == b.get("category_name"):
                iou = compute_box_iou(bbox, b.get("bbox", []))
                if iou > 0.95:
                    is_anomaly = True
                    reasons.append(f"Duplicate box with IoU {iou:.2f}")
                    break
                    
        # Rule 7: Loose shelf padding
        if w > 0.65 * img_w and h < 0.08 * img_h:
            is_anomaly = True
            reasons.append("Loose shelf padding across multiple bays")
            
        # Rule 8: Dense cluster overlaps (excessive overlap with multiple boxes)
        overlapping_count = sum(1 for j, b in enumerate(anns) if i != j and compute_box_iou(bbox, b.get("bbox", [])) > 0.70)
        if overlapping_count >= 3:
            is_anomaly = True
            reasons.append(f"Dense cluster overlap ({overlapping_count} overlapping boxes)")
            
        if is_anomaly:
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": reasons,
                "bbox": bbox,
                "category_name": a.get("category_name", "Unknown")
            })
        else:
            clean_annotations.append(a)

logger.info("=" * 70)
logger.info(f"ANOMALY AUDIT COMPLETE: {len(flagged_anomalies)} / {total_boxes} boxes flagged ({len(flagged_anomalies)/max(1, total_boxes)*100:.2f}%)")
logger.info(f"Clean annotations retained: {len(clean_annotations)}")
logger.info("=" * 70)

In [ ]:
# CELL 6 - Comprehensive Anomaly Diagnostic Graphs
rule_counts = Counter()
for item in flagged_anomalies:
    for r in item["reasons"]:
        rule_counts[r.split("(")[0].strip()] += 1

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Anomaly Rule Frequency Breakdown
labels = list(rule_counts.keys()) or ["No Anomalies"]
counts = list(rule_counts.values()) or [0]
bars = axs[0, 0].barh(labels, counts, color="#E53935")
axs[0, 0].set_title("Flagged Anomalies by Rule Category", fontsize=12, fontweight="bold")
axs[0, 0].set_xlabel("Count")
for b in bars:
    axs[0, 0].text(b.get_width() + 0.5, b.get_y() + b.get_height()/2.0, str(int(b.get_width())), va="center", fontsize=10)

# Panel 2: Aspect Ratio Comparison: Clean vs Anomaly
clean_aspects = [a["bbox"][2] / max(1.0, a["bbox"][3]) for a in clean_annotations if len(a.get("bbox", [])) == 4]
anom_aspects = [a["bbox"][2] / max(1.0, a["bbox"][3]) for a in flagged_anomalies if len(a.get("bbox", [])) == 4]

axs[0, 1].hist(clean_aspects, bins=30, range=(0, 10), alpha=0.7, label=f"Clean ({len(clean_aspects)})", color="#43A047")
if anom_aspects:
    axs[0, 1].hist(anom_aspects, bins=30, range=(0, 10), alpha=0.7, label=f"Anomalies ({len(anom_aspects)})", color="#E53935")
axs[0, 1].set_title("Aspect Ratio Distribution: Clean vs Anomalies", fontsize=12, fontweight="bold")
axs[0, 1].set_xlabel("Aspect Ratio (Width / Height)")
axs[0, 1].axvline(0.1, color="red", linestyle="--", label="Sliver Cutoff (<0.1)")
axs[0, 1].legend()

# Panel 3: Box Area (Log-scale) Comparison
clean_areas = [a["bbox"][2] * a["bbox"][3] for a in clean_annotations if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]
anom_areas = [a["bbox"][2] * a["bbox"][3] for a in flagged_anomalies if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]

axs[1, 0].hist(np.log10(clean_areas), bins=30, alpha=0.7, label="Clean", color="#1E88E5")
if anom_areas:
    axs[1, 0].hist(np.log10(anom_areas), bins=30, alpha=0.7, label="Anomalies", color="#E53935")
axs[1, 0].set_title("Bounding Box Area Distribution (log10 px²)", fontsize=12, fontweight="bold")
axs[1, 0].set_xlabel("log10(Area)")
axs[1, 0].legend()

# Panel 4: Quality Retention Overview Pie Chart
clean_count = len(clean_annotations)
flagged_count = len(flagged_anomalies)
if total_boxes > 0:
    axs[1, 1].pie([clean_count, flagged_count], labels=[f"Clean ({clean_count})", f"Flagged ({flagged_count})"], autopct="%1.1f%%", colors=["#43A047", "#E53935"], startangle=140, explode=(0, 0.1))
    axs[1, 1].set_title("Dataset Quality Split (Retained vs Excluded)", fontsize=12, fontweight="bold")

plt.tight_layout()
show_plt_figure(fig)

In [ ]:
# CELL 7 - Visual Display of Detected Anomaly Samples (Side-by-Side Overlays)
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display

def render_anomaly_samples(anomalies_list, images_dir: Path, max_samples: int = 6):
    """Renders visual inspection images highlighting flagged anomalies (red) vs clean tags (green)."""
    if not anomalies_list:
        logger.info("✓ Zero anomalies detected to preview.")
        return
        
    logger.info(f"Rendering {min(max_samples, len(anomalies_list))} detected anomaly case previews...")
    # Group by image
    anom_by_img = defaultdict(list)
    for a in anomalies_list:
        anom_by_img[a["image_url"]].append(a)
        
    previewed = 0
    for img_url, a_list in anom_by_img.items():
        if previewed >= max_samples:
            break
        img_name = Path(img_url).name
        search_dirs = [
            str(images_dir),
            str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
            str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
            str(Path("coco_files/images")),
            str(Path("images")),
            "/home/jupyter/multi_class_train_rfdetr/images",
            "/home/jupyter/coco_files/images"
        ]
        candidate_names = [img_name, Path(img_name).name, f"{Path(img_name).stem}.jpg"]
        img_path = None
        for d in search_dirs:
            if not d or not os.path.exists(d): continue
            for name in candidate_names:
                p = os.path.join(d, name)
                if os.path.exists(p) and os.path.getsize(p) > 0:
                    img_path = Path(p)
                    break
            if img_path: break
        if img_path is None:
            img_path = images_dir / img_name
        if not img_path.exists():
            continue
            
        img = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        
        # Draw clean boxes in green
        img_ann_lookup = annotations_by_image if 'annotations_by_image' in globals() else image_annotations
        clean_in_img = img_ann_lookup.get(img_url, [])
        for c in clean_in_img:
            if c not in [a["annotation"] for a in a_list]:
                cb = c.get("bbox", [])
                if len(cb) == 4:
                    draw.rectangle([cb[0], cb[1], cb[0]+cb[2], cb[1]+cb[3]], outline="#00E676", width=2)
                    
        # Draw anomaly boxes in red with yellow reason badge
        for anom in a_list:
            ab = anom["bbox"]
            draw.rectangle([ab[0], ab[1], ab[0]+ab[2], ab[1]+ab[3]], outline="#D50000", width=4)
            reason_txt = " | ".join(anom["reasons"])
            draw.rectangle([ab[0], max(0, ab[1]-22), ab[0]+len(reason_txt)*8, ab[1]], fill="#FFD600")
            draw.text((ab[0]+4, max(2, ab[1]-18)), reason_txt, fill="#000000")
            
        logger.info(f"Preview {previewed+1}: {img_name} - Flagged: {len(a_list)} anomalies")
        # Resize for display
        display(img.resize((img.width // 2, img.height // 2)))
        previewed += 1

render_anomaly_samples(flagged_anomalies, IMAGES_DIR, max_samples=4)

In [ ]:
# CELL 8 - Export Manifest & Sanitized Dataset
manifest_path = OUTPUT_ANOMALIES_DIR / "anomalies_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "flagged_count": len(flagged_anomalies),
        "total_audited": total_boxes,
        "clean_count": len(clean_annotations),
        "anomalies": flagged_anomalies
    }, f, indent=2)

logger.info(f"✓ Saved anomalies manifest: {manifest_path}")

clean_coco = {
    "images": list({a.get("image_id"): {"id": a.get("image_id"), "file_name": Path(a.get("image_url", "")).name or f"{a.get('image_id')}.jpg", "coco_url": a.get("image_url")} for a in clean_annotations}.values()),
    "annotations": clean_annotations,
    "categories": categories_list
}
clean_json_path = CLEANED_DATASET_DIR / "_annotations.coco.json"
with open(clean_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_coco, f, indent=2)

logger.info(f"✓ Saved sanitized dataset: {clean_json_path} ({len(clean_coco['annotations'])} clean annotations)")

In [ ]:
# CELL 9 - Step B Verification on Cleaned Dataset
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Verify sanitized output file exists
target_cleaned_ann = clean_json_path if 'clean_json_path' in globals() else CLEANED_DATASET_DIR / "_annotations.coco.json"
if target_cleaned_ann.exists():
    cleaned_info = load_coco_info(str(target_cleaned_ann))
    logger.info("=" * 65)
    logger.info(f"Cleaned Dataset Audit: {cleaned_info['num_images']} images, {cleaned_info['num_anns']} annotations, {cleaned_info['num_classes']} classes")
    logger.info(f"Classes: {cleaned_info['categories']}")
    logger.info("=" * 65)
else:
    logger.info(f"Cleaned dataset file {target_cleaned_ann} not generated yet.")
